In [0]:
df_bronze = spark.table('retail_project_dev.bronze.raw_sales')

In [0]:
%sql
DESCRIBE TABLE retail_project_dev.bronze.raw_sales

In [0]:
%sql
select * from retail_project_dev.bronze.raw_sales
limit 10

In [0]:
from pyspark.sql.functions import col, max

# Returns True if a column has ANY nulls, False otherwise
has_nulls_df = spark.table('retail_project_dev.bronze.raw_sales').select([max(col(c).isNull()).alias(c) for c in spark.table('retail_project_dev.bronze.raw_sales').columns])

display(has_nulls_df)

# Drop NA in the critical columns

In [0]:
critical_columns = ['TransactionID', 'ProductID', 'Quantity', 'Amount', 'TransactionDate']

df_clean1 = df_bronze.dropna(subset=critical_columns)
display(df_clean1)

# Check if there are dublicates in transactionID col

In [0]:
from pyspark.sql.functions import col
duplicate_ids = df_bronze.groupBy("TransactionID").count().filter(col("count") > 1)
display(duplicate_ids)

## Check for rows where 'amount' contains anything OTHER than numbers and decimals

In [0]:
from pyspark.sql.functions import col
 
df_wierdSybols = df_clean1.filter(col("Amount").rlike("[^0-9.]"))
display(df_wierdSybols)

# Cast proper datatype

In [0]:
df_clean2 = (
    df_clean1
    .withColumn('customerage', col("CustomerAge").cast('int'))
    .withColumn('TranscationDate', col("TransactionDate").cast('timestamp'))
    .withColumn('Quantity', col("Quantity").cast('int'))
    .withColumn('Amount', col("Amount").cast('float'))
    .withColumn('Discount', col("Discount").cast('float'))
)

# Standarize columns

In [0]:
from pyspark.sql.functions import upper, trim, col
df_clean3 = (
    df_clean2
    .withColumn("PaymentType", upper(trim(col("PaymentType")))) \
    .withColumn("StoreRegion", upper(trim(col("StoreRegion")))) \
    .withColumn("DeviceUsed", upper(trim(col("DeviceUsed"))))
)

In [0]:
display(df_clean3.limit(5))

In [0]:
df_clean3.write.mode("overwrite").saveAsTable("retail_project_dev.silver.sales_clean")  